In [ ]:
RUN_TARGET = "local"  # "colab" | "local"


## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1.
2. Run the install cell and restart if requested.
3. Mount Drive if you want to read synced `models/` and `results/` artifacts.
4. Run the remaining cells top to bottom.

### Running locally
1. Keep `RUN_TARGET = "local"`.
2. The notebook scans the local `models/` and `results/` folders.
3. It loads saved probe artifacts and renders main-paper and supplementary figures without retraining or reprobe computation.


In [ ]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "scipy": "1.15.3",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "statsmodels": "0.14.5",
        "seaborn": "0.13.2",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


# 07 - Cross-Model Probe Visualisation

This notebook is the centralized visualization layer for probe outputs that already exist in `results/`. It does not load models or generate attribution scores; it only discovers saved probe-row CSVs, combines them, and renders main-paper plus supplementary figures.


In [ ]:
import sys
from pathlib import Path

# Allow imports from the src-layout package without a root-level shim.
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break
import importlib

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils

importlib.reload(baseline_notebook_utils)
importlib.reload(mtl_epitope_notebook_utils)

from xallergen.baseline_notebook_utils import configure_matplotlib_cache, detect_device, find_project_root, print_runtime_context
from xallergen.mtl_epitope_notebook_utils import discover_probe_row_artifacts, load_available_probe_rows, render_probe_figures_from_rows, save_combined_probe_tables

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd


In [ ]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


## Discover Available Variants

Each compatible checkpoint in `models/` is mapped to the probe-row CSV that should already exist in `results/`. If a probe artifact is missing, this notebook skips that model rather than recomputing it. Use `06_generate_probe_rows.ipynb` to generate or refresh missing probe artifacts first.


In [ ]:
discovery_df = discover_probe_row_artifacts(MODELS_DIR, RESULTS_DIR)
discovery_df


In [ ]:
all_probe_df = load_available_probe_rows(discovery_df)
all_probe_df.head()


## Save Combined Tables

These artifacts provide a single reusable row-wise table plus bootstrap-CI summaries across all available families. Figures can be regenerated later from `all_models_probing_rows.csv` without rerunning training or probing.

In [ ]:
table_outputs = save_combined_probe_tables(all_probe_df, RESULTS_DIR)
MULTI_MODEL_ROWS_CSV = table_outputs["rows_path"]
MULTI_MODEL_SUMMARY_CSV = table_outputs["summary_path"]
summary_df = table_outputs["summary_df"]
summary_df


## Plot All Available Families

In [ ]:
figure_outputs = render_probe_figures_from_rows(all_probe_df, RESULTS_DIR)
figure_outputs


In [ ]:
print("Saved combined artifacts:")
for out_path in [MULTI_MODEL_ROWS_CSV, MULTI_MODEL_SUMMARY_CSV]:
    print(f"  {out_path}")
for label, out_path in figure_outputs.items():
    print(f"  {label}: {out_path}")
